# CNN for CIFAR 10 dataset

In [ ]:
# 32 x 32 x 3 -> colored images (60000 thousands images)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [ ]:
# datasets and dataloaders

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image -> scale (0,1) => normalize (-1,1)
transform = transforms.Compose([# helps in chaining
    transforms.ToTensor(), # image to tensors and auto scale
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # std, mean
])

print("downloading data.....")
trainset = CIFAR10(root="./data",train=True, download=True, transform=transform)
testset = CIFAR10(root="./data",train=False, download=True, transform=transform)

In [ ]:
trainset

In [ ]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

## build the CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(

            # 1st convolutional layer + pooling
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel size = 2, stride = 2

            # 2nd Conv + pool layer
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
            #softmax A.F automatically by cross entropy loss
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening step
        x = self.fc_layers(x)

        return x

In [ ]:
model = CNN()

In [ ]:
crietrion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# training the CNN

epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    model.train()
    for images, labels in trainloader:
        optimizer.zero_grad()
        
        outputs = model.forward(images)
        loss = crietrion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch {epoch+1}/{epochs}, loss = {epoch_training_loss/len(trainloader)}")